# SPAI v5 — train way better detectors

Runs the full v5 recipe on a free **Colab T4**: clean the labels, train YOLO11s with SPAI-tuned augmentation, compare against the shipped model on the same split, and export `.pt` + CoreML.

See `model-training/v5/README.md` for why each step exists.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics
import ultralytics; ultralytics.checks()

In [ ]:
# Get the repo (scripts + current weights to compare against).
!git clone -q https://github.com/juanadams362-NYC/SPAI.git
%cd /content/SPAI/model-training/v5

## 1 · Data
Upload or download your merged PPE dataset (YOLO format: `images/{train,val,test}`, `labels/...`, `data.yaml` with `0: glove`, `1: hand`) to `/content/merged`. Roboflow: *Export → YOLOv8 → show download code*.

In [ ]:
PPE_DATA = '/content/merged'          # change if needed
INSTRUMENT_DATA = '/content/ASTRA'    # ASTRA Surgical Instruments v12 (YOLOv8 export)

## 2 · Audit and clean the labels
Finds boxes labeled both glove and hand, writes a cleaned copy, and lists probable missing labels for review.

In [ ]:
!python audit_labels.py $PPE_DATA --fix --out /content/merged_clean
!python audit_labels.py $PPE_DATA --model ../../backend/model/best.pt --conf 0.5
import pandas as pd
display(pd.read_csv(f'{PPE_DATA}/missing_labels.csv').head(20))
display(pd.read_csv(f'{PPE_DATA}/class_disagreements.csv').head(20))

## 3 · Train PPE v5 (≈1.5 h)
The script prints a side-by-side table against the current model and only says *ship it* if v5 wins.

In [ ]:
!python train_v5.py --task ppe --data /content/merged_clean/data.yaml --baseline ../../backend/model/best.pt --export

## 4 · Train instruments v5 (≈2 h)

In [ ]:
!python train_v5.py --task instruments --data $INSTRUMENT_DATA/data.yaml --baseline ../../backend/model/instruments_best.pt --export

## 5 · Save the results
Copy the exports to Drive, then into the repo (see README step 4).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/SPAI_v5 && cp -r runs/v5/*/export /content/drive/MyDrive/SPAI_v5/
!ls -R /content/drive/MyDrive/SPAI_v5